<a href="https://colab.research.google.com/github/2303A52151/reinforcement_learning_2151/blob/main/lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# 1. Define the MDP
non_terminal_states = ["S0", "S1"]
terminal_states = ["G"]
all_states = non_terminal_states + terminal_states
actions = ["Up", "Down", "Left", "Right"]

gamma = 0.9
theta = 1e-6

# Deterministic transitions: transitions[state][action] = (next_state, reward)
transitions = {
    "S0": {
        "Up":    ("S0", -1),
        "Down":  ("S1", -1),
        "Left":  ("S0", -1),
        "Right": ("S0", -1),   # Wall
    },
    "S1": {
        "Up":    ("S0", -1),
        "Down":  ("S1", -1),
        "Left":  ("S1", -1),
        "Right": ("G", 100),   # Goal
    }
}

# Print State, Action, Reward, and Transition information
mdp_rows = []
for s in non_terminal_states:
    for a in actions:
        s_next, r = transitions[s][a]
        mdp_rows.append({"State": s, "Action": a, "Next State": s_next, "Reward": r})

print("=== MDP Transition & Reward Table ===")
print(pd.DataFrame(mdp_rows).to_string(index=False))

# 2. Initialize state values to zero
V = {s: 0.0 for s in all_states}
history = [{"Iteration": 0, "V(S0)": V["S0"], "V(S1)": V["S1"], "V(G)": V["G"], "Delta": None}]

# 3-5. Value Iteration loop
k = 0
while True:
    k += 1
    delta = 0.0
    V_new = V.copy()

    for s in non_terminal_states:
        q_values = [transitions[s][a][1] + gamma * V[transitions[s][a][0]] for a in actions]
        V_new[s] = max(q_values)
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    history.append({
        "Iteration": k,
        "V(S0)": round(V["S0"], 4),
        "V(S1)": round(V["S1"], 4),
        "V(G)": round(V["G"], 4),
        "Delta": round(delta, 6)
    })

    if delta < theta:
        break

# 6. Display final results
print("\n=== Iteration-Wise State-Value Table ===")
print(pd.DataFrame(history).to_string(index=False))

print("\n=== Final Optimal State Values V*(s) ===")
for s in all_states:
    print(f"V*({s}) = {V[s]:.2f}")

q_rows = []
optimal_policy = {}

for s in non_terminal_states:
    best_a, best_q = None, float("-inf")
    for a in actions:
        s_next, r = transitions[s][a]
        q_val = round(r + gamma * V[s_next], 4)
        q_rows.append({"State": s, "Action": a, "Next State": s_next, "Q*(s, a)": q_val})
        if q_val > best_q:
            best_q, best_a = q_val, a
    optimal_policy[s] = best_a

print("\n=== Final Action-Value Table Q*(s, a) ===")
print(pd.DataFrame(q_rows).to_string(index=False))

print("\n=== Optimal Policy pi*(s) ===")
for s, a in optimal_policy.items():
    print(f"pi*({s}) = {a} (V*({s}) = {V[s]:.2f})")

# Trace optimal path
path = ["S0"]
curr = "S0"
while curr not in terminal_states:
    curr = transitions[curr][optimal_policy[curr]][0]
    path.append(curr)

print("\nOptimal Path:", " -> ".join(path))

=== MDP Transition & Reward Table ===
State Action Next State  Reward
   S0     Up         S0      -1
   S0   Down         S1      -1
   S0   Left         S0      -1
   S0  Right         S0      -1
   S1     Up         S0      -1
   S1   Down         S1      -1
   S1   Left         S1      -1
   S1  Right          G     100

=== Iteration-Wise State-Value Table ===
 Iteration  V(S0)  V(S1)  V(G)  Delta
         0    0.0    0.0   0.0    NaN
         1   -1.0  100.0   0.0  100.0
         2   89.0  100.0   0.0   90.0
         3   89.0  100.0   0.0    0.0

=== Final Optimal State Values V*(s) ===
V*(S0) = 89.00
V*(S1) = 100.00
V*(G) = 0.00

=== Final Action-Value Table Q*(s, a) ===
State Action Next State  Q*(s, a)
   S0     Up         S0      79.1
   S0   Down         S1      89.0
   S0   Left         S0      79.1
   S0  Right         S0      79.1
   S1     Up         S0      79.1
   S1   Down         S1      89.0
   S1   Left         S1      89.0
   S1  Right          G     100.0

=== Op